<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>
<h2 align="center">Roteiro de Atividade Prática</h2>
<br>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Análise Exploratória de Dados (U1)  
**Semana:** 8 — Tratamento de Dados Ausentes e Outliers  
**Aula 4:** Tratamento de problemas de qualidade dos dados  
**Tipo:** Prática | **Bloom:** Analisar


# Auditoria de qualidade da base semanal de pedidos

## Situação profissional

Vocês atuam como **assistentes de dados da equipe de operações** de um centro de distribuição de comércio eletrônico. A equipe precisa preparar a base semanal de pedidos antes de produzir indicadores operacionais.

A base contém ausências e valores muito diferentes dos demais. O desafio não é “limpar tudo”: é decidir **o que deve ser mantido, corrigido, tratado como ausente ou investigado**, usando evidências e regras de negócio.

**Missão:** analisar as ocorrências, registrar decisões justificadas, aplicar o plano em uma cópia da base e verificar como as intervenções afetam os resumos utilizados na análise.

> Os dados são **simulados de alta fidelidade** e foram construídos para fins didáticos. Eles reproduzem estrutura e problemas plausíveis de uma operação de e-commerce, mas não representam uma empresa real.


## Preparação

Arquivos usados nesta atividade:

- o notebook desta atividade;
- `DADOS3EMC3B2S8A4_pedidos_operacao.csv`;
- `DADOS3EMC3B2S8A4_verificacoes_qualidade.csv`.

Coloquem os arquivos na mesma pasta e executem as células em ordem. O código é um **instrumento para analisar e verificar as decisões**; a tarefa principal é interpretar evidências e justificar o tratamento.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display


def localizar(nome):
    candidatos = [Path(nome), Path('/mnt/data') / nome]
    for p in candidatos:
        if p.is_file():
            return p
    raise FileNotFoundError(f"Arquivo não encontrado: {nome}. Coloque os arquivos na mesma pasta do notebook.")

arquivo_dados = localizar("DADOS3EMC3B2S8A4_pedidos_operacao.csv")
arquivo_verificacoes = localizar("DADOS3EMC3B2S8A4_verificacoes_qualidade.csv")

dados = pd.read_csv(arquivo_dados, encoding="utf-8-sig")
verificacoes = pd.read_csv(arquivo_verificacoes, encoding="utf-8-sig")

dados["data_pedido"] = pd.to_datetime(dados["data_pedido"])

print(f"Registros: {len(dados)} | Colunas: {dados.shape[1]}")
display(dados.head())


## Etapa 1 — Reconheçam a base e o escopo da auditoria

Cada registro representa **um pedido processado em uma operação de e-commerce**.

| Campo | Significado |
|---|---|
| `pedido_id` | Identificador do pedido. |
| `data_pedido` | Data e hora de entrada do pedido. |
| `centro_distribuicao` | Centro responsável pelo processamento. |
| `turno` | Turno operacional. |
| `canal_venda` | Site, App ou Marketplace. |
| `categoria_principal` | Categoria predominante do pedido. |
| `modalidade_entrega` | Entrega padrão, expressa ou retirada. |
| `itens_qtd` | Quantidade de itens do pedido. |
| `valor_pedido_reais` | Valor total do pedido, em reais. |
| `cupom_aplicado` | Indica se houve cupom. |
| `desconto_pct` | Percentual de desconto registrado. |
| `distancia_km` | Distância de entrega; não se aplica a retirada. |
| `tempo_separacao_min` | Tempo entre início e fim da separação, em minutos. |
| `avaliacao_cliente_1a5` | Nota voluntária de 1 a 5 após a entrega. |

Nesta atividade, a equipe prioriza **ausências** e possíveis outliers nos campos `tempo_separacao_min` e `itens_qtd`. Outros campos permanecem disponíveis para interpretar o contexto dos pedidos.

### Registro 1

1. Por que uma ausência em `distancia_km` pode ter significado diferente de uma ausência em `tempo_separacao_min`?  
**Resposta:**

2. Por que um valor muito alto em `tempo_separacao_min` não deve ser removido automaticamente?  
**Resposta:**


## Etapa 2 — Façam a triagem das ausências

Executem a célula abaixo. Observem **quantidade e proporção** de ausências por campo. Em seguida, consultem as regras gerais em `verificacoes` (`escopo = regra`).


In [ ]:
faltantes = pd.DataFrame({
    "quantidade_ausente": dados.isna().sum(),
    "percentual_ausente": (dados.isna().mean() * 100).round(1)
}).query("quantidade_ausente > 0").sort_values("quantidade_ausente", ascending=False)

display(faltantes)
print("Regras gerais disponíveis:")
display(verificacoes.query("escopo == 'regra'")[[
    "evidencia_id", "campo", "condicao_observada", "resultado_verificacao", "fonte_operacional"
]])


### Registro 2 — Classifiquem as ausências

Para cada situação abaixo, registrem uma decisão inicial e a evidência que a sustenta.

| Situação | Decisão da dupla | Evidência usada |
|---|---|---|
| `desconto_pct` ausente quando `cupom_aplicado = Não` |  |  |
| `distancia_km` ausente em pedidos de retirada |  |  |
| `avaliacao_cliente_1a5` ausente |  |  |
| `tempo_separacao_min` ausente em `PED-2026-0056` |  |  |
| `tempo_separacao_min` ausente em `PED-2026-0094` |  |  |

> Lembrem-se: ausência não significa automaticamente zero e não torna automaticamente todo o registro inútil.


## Etapa 3 — Identifiquem possíveis outliers nos campos priorizados

A função abaixo usa o critério de **1,5 × AIQ**, já trabalhado, para localizar candidatos. O resultado indica **o que precisa ser verificado**, não o que deve ser excluído.


In [ ]:
def candidatos_iqr(df, coluna):
    s = df[coluna].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    aiq = q3 - q1
    limite_inferior = q1 - 1.5 * aiq
    limite_superior = q3 + 1.5 * aiq
    mask = (df[coluna] < limite_inferior) | (df[coluna] > limite_superior)
    candidatos = df.loc[mask, ["pedido_id", coluna, "categoria_principal", "modalidade_entrega", "valor_pedido_reais"]].copy()
    return candidatos, q1, q3, limite_inferior, limite_superior

for coluna in ["tempo_separacao_min", "itens_qtd"]:
    candidatos, q1, q3, li, ls = candidatos_iqr(dados, coluna)
    print(f"\n{coluna}: Q1={q1:.1f} | Q3={q3:.1f} | limite inferior={li:.1f} | limite superior={ls:.1f}")
    display(candidatos)

fig, ax = plt.subplots(figsize=(5.5, 5.2))
ax.boxplot(dados["tempo_separacao_min"].dropna())
ax.set_title("Tempo de separação — antes do tratamento")
ax.set_ylabel("Tempo de separação (minutos)")
ax.set_xticks([1])
ax.set_xticklabels(["tempo_separacao_min"])
plt.show()


Agora consultem as verificações específicas dos pedidos sinalizados. Leiam o resultado e a fonte operacional antes de decidir.


In [ ]:
casos = verificacoes.query("escopo == 'pedido'")[[
    "evidencia_id", "pedido_id", "campo", "condicao_observada",
    "resultado_verificacao", "valor_confirmado", "fonte_operacional"
]].copy()
display(casos)


### Registro 3 — Montem o plano de tratamento

Para cada item abaixo, consultem a evidência disponível, escolham uma ação e registrem a justificativa **antes de aplicar qualquer alteração**.

Usem apenas ações compatíveis com o que foi estudado:

- `MANTER_AUSENTE`
- `PREENCHER_ZERO`
- `CORRIGIR_VALOR_CONFIRMADO`
- `MANTER`
- `TRATAR_COMO_AUSENTE`
- `INVESTIGAR_MAIS`

Preencham a tabela abaixo. Ela registra o raciocínio da dupla; o log automático da etapa seguinte registrará somente as intervenções efetivamente executadas.

| Item/pedido | Campo | Evidência consultada | Ação escolhida | Justificativa |
|---|---|---|---|---|
| `REGRA_DESCONTO_SEM_CUPOM` | `desconto_pct` |  |  |  |
| `REGRA_DISTANCIA_RETIRADA` | `distancia_km` |  |  |  |
| `REGRA_AVALIACAO_SEM_RESPOSTA` | `avaliacao_cliente_1a5` |  |  |  |
| `PED-2026-0056` | `tempo_separacao_min` |  |  |  |
| `PED-2026-0094` | `tempo_separacao_min` |  |  |  |
| `PED-2026-0047` | `tempo_separacao_min` |  |  |  |
| `PED-2026-0112` | `itens_qtd` |  |  |  |
| `PED-2026-0159` | `tempo_separacao_min` |  |  |  |
| `PED-2026-0136` | `tempo_separacao_min` |  |  |  |
| `PED-2026-0171` | `tempo_separacao_min` |  |  |  |

Depois de completar a tabela, preencham **somente os textos entre aspas vazias** na célula de código seguinte. A validação interromperá a execução se alguma ação estiver em branco ou não for compatível com o item analisado.


In [ ]:
# Preencham as ações somente depois de completar a tabela de decisões acima.
plano = {
    "REGRA_DESCONTO_SEM_CUPOM": "",
    "REGRA_DISTANCIA_RETIRADA": "",
    "REGRA_AVALIACAO_SEM_RESPOSTA": "",
    "PED-2026-0056": "",
    "PED-2026-0094": "",
    "PED-2026-0047": "",
    "PED-2026-0112": "",
    "PED-2026-0159": "",
    "PED-2026-0136": "",
    "PED-2026-0171": "",
}

# Cada item aceita apenas ações que fazem sentido para a evidência disponível
# e que possuem comportamento explicitamente implementado no notebook.
permitidas_por_item = {
    "REGRA_DESCONTO_SEM_CUPOM": {"PREENCHER_ZERO"},
    "REGRA_DISTANCIA_RETIRADA": {"MANTER_AUSENTE"},
    "REGRA_AVALIACAO_SEM_RESPOSTA": {"MANTER_AUSENTE"},
    "PED-2026-0056": {"CORRIGIR_VALOR_CONFIRMADO"},
    "PED-2026-0094": {"MANTER_AUSENTE"},
    "PED-2026-0047": {"CORRIGIR_VALOR_CONFIRMADO"},
    "PED-2026-0112": {"CORRIGIR_VALOR_CONFIRMADO"},
    "PED-2026-0159": {"MANTER"},
    "PED-2026-0136": {"TRATAR_COMO_AUSENTE"},
    "PED-2026-0171": {"INVESTIGAR_MAIS"},
}


def validar_plano(plano):
    problemas = []
    for item, acao in plano.items():
        if not acao:
            problemas.append({
                "item": item,
                "ação_registrada": "(não preenchida)",
                "problema": "decisão ainda não registrada"
            })
        elif acao not in permitidas_por_item[item]:
            problemas.append({
                "item": item,
                "ação_registrada": acao,
                "problema": "ação incompatível com este item; revise a evidência e a justificativa"
            })
    return pd.DataFrame(problemas)


problemas_plano = validar_plano(plano)

if not problemas_plano.empty:
    print(f"Plano ainda não validado: {len(problemas_plano)} item(ns) precisam ser revistos.")
    display(problemas_plano)
    print("Revisem a evidência de cada item, corrijam somente os textos entre aspas e executem esta célula novamente.")
else:
    print("Plano validado. Avancem para a Etapa 4 e apliquem o plano em uma cópia da base.")


## Etapa 4 — Apliquem o plano em uma cópia e preservem o original

A célula abaixo **não altera `dados`**. Ela cria `dados_tratados` e aplica somente as ações validadas em `plano`. Se houver decisão não preenchida ou ação incompatível com o item analisado, a aplicação será interrompida e o problema será indicado.


In [ ]:
def valor_confirmado(pedido_id, campo):
    linha = verificacoes.query("pedido_id == @pedido_id and campo == @campo")
    if linha.empty or pd.isna(linha.iloc[0]["valor_confirmado"]):
        return np.nan
    return float(linha.iloc[0]["valor_confirmado"])


def aplicar_plano(dados_originais, plano):
    problemas = validar_plano(plano)
    if not problemas.empty:
        print("Aplicação não executada: há decisões não preenchidas ou incompatíveis com os itens analisados.")
        display(problemas)
        return None

    df = dados_originais.copy()
    intervencoes = []

    def registrar_intervencao(item, campo, antes, depois, acao):
        intervencoes.append({
            "item": item,
            "campo": campo,
            "antes": antes,
            "depois": depois,
            "acao": acao,
        })

    # Regra geral: ausência de desconto quando não houve cupom representa 0%.
    item = "REGRA_DESCONTO_SEM_CUPOM"
    mask = df["cupom_aplicado"].eq("Não") & df["desconto_pct"].isna()
    antes = int(mask.sum())
    df.loc[mask, "desconto_pct"] = 0.0
    depois = int(df.loc[mask, "desconto_pct"].isna().sum())
    registrar_intervencao(item, "desconto_pct", f"{antes} ausentes", f"{depois} ausentes", plano[item])

    # As duas decisões abaixo são registradas na tabela, mas não alteram a base:
    # distância não se aplica à retirada; avaliação sem resposta permanece ausente.

    mapa_campos = {
        "PED-2026-0056": "tempo_separacao_min",
        "PED-2026-0094": "tempo_separacao_min",
        "PED-2026-0047": "tempo_separacao_min",
        "PED-2026-0112": "itens_qtd",
        "PED-2026-0159": "tempo_separacao_min",
        "PED-2026-0136": "tempo_separacao_min",
        "PED-2026-0171": "tempo_separacao_min",
    }

    for pedido_id, campo in mapa_campos.items():
        idx = df.index[df["pedido_id"].eq(pedido_id)]
        if len(idx) != 1:
            raise ValueError(f"Pedido não encontrado ou duplicado: {pedido_id}")
        idx = idx[0]
        antes = df.at[idx, campo]
        acao = plano[pedido_id]

        if acao == "CORRIGIR_VALOR_CONFIRMADO":
            novo = valor_confirmado(pedido_id, campo)
            if pd.isna(novo):
                raise ValueError(f"Não há valor confirmado para {pedido_id} / {campo}")
            if campo == "itens_qtd":
                novo = int(novo)
            df.at[idx, campo] = novo
            registrar_intervencao(pedido_id, campo, antes, df.at[idx, campo], acao)

        elif acao == "TRATAR_COMO_AUSENTE":
            df.at[idx, campo] = np.nan
            registrar_intervencao(pedido_id, campo, antes, df.at[idx, campo], acao)

        elif acao in {"MANTER", "MANTER_AUSENTE", "INVESTIGAR_MAIS"}:
            # A decisão está documentada na tabela de decisões; não há intervenção a executar.
            pass

        else:
            # Salvaguarda: esta condição não deve ser alcançada após validar_plano().
            raise ValueError(f"Ação sem implementação para {pedido_id}: {acao}")

    return df, pd.DataFrame(intervencoes)


# Reiniciamos estas variáveis a cada execução para evitar usar uma base tratada de uma execução anterior.
dados_tratados = None
log_auditoria = None

resultado = aplicar_plano(dados, plano)
if resultado is not None:
    dados_tratados, log_auditoria = resultado
    print("Plano aplicado em uma cópia da base.")
    print("Log automático das intervenções efetivamente executadas:")
    display(log_auditoria)
    print("As decisões de manter ou investigar permanecem documentadas na tabela do Registro 3 e não aparecem como intervenções no log.")
else:
    print("Retornem à Etapa 3, revisem o plano e executem novamente as Etapas 3 e 4 antes de comparar os resultados.")


## Etapa 5 — Verifiquem as consequências

Depois de completar o plano e executar a aplicação, comparem **antes e depois**. O objetivo não é fazer desaparecer todos os valores atípicos nem zerar todas as ausências; é verificar se cada intervenção foi coerente com a evidência disponível.

> **Importante:** as comparações abaixo só são geradas depois que o plano estiver completo e a Etapa 4 tiver criado `dados_tratados`.

In [ ]:
if "dados_tratados" not in globals() or dados_tratados is None:
    print("Comparação ainda não gerada: nesta execução, a base tratada não foi criada.")
    problemas_plano = validar_plano(plano)
    if not problemas_plano.empty:
        print(f"Há {len(problemas_plano)} item(ns) a revisar no plano:")
        display(problemas_plano)
    print("Para gerar a comparação: validem o plano na Etapa 3, executem novamente a Etapa 4 e, depois, esta célula.")
else:
    print("Comparação entre a base original e a cópia tratada:")

    comparacao_faltantes = pd.DataFrame({
        "antes": dados.isna().sum(),
        "depois": dados_tratados.isna().sum(),
    })
    comparacao_faltantes = comparacao_faltantes.loc[(comparacao_faltantes != 0).any(axis=1)]
    print("\nAusências antes e depois:")
    display(comparacao_faltantes)

    resumo = pd.DataFrame({
        "antes": [dados["tempo_separacao_min"].mean(), dados["tempo_separacao_min"].median()],
        "depois": [dados_tratados["tempo_separacao_min"].mean(), dados_tratados["tempo_separacao_min"].median()],
    }, index=["média_tempo_separacao", "mediana_tempo_separacao"]).round(2)
    print("Resumo do tempo de separação:")
    display(resumo)

    for coluna in ["tempo_separacao_min", "itens_qtd"]:
        candidatos, q1, q3, li, ls = candidatos_iqr(dados_tratados, coluna)
        print(f"\n{coluna} após tratamento: limite superior={ls:.1f}")
        display(candidatos)

    fig, ax = plt.subplots(figsize=(5.5, 5.2))
    ax.boxplot(dados_tratados["tempo_separacao_min"].dropna())
    ax.set_title("Tempo de separação — após o plano de tratamento")
    ax.set_ylabel("Tempo de separação (minutos)")
    ax.set_xticks([1])
    ax.set_xticklabels(["tempo_separacao_min"])
    plt.show()


### Registro 4 — Conclusão profissional

Respondam de forma breve:

1. Quais ausências permaneceram na base e por que **não deveriam ser preenchidas automaticamente**?  
**Resposta:**

2. Quais possíveis outliers permaneceram após o tratamento? Explique por que a permanência deles não significa que o tratamento falhou.  
**Resposta:**

3. Qual caso permanece sem informação suficiente para uma decisão definitiva? O que a equipe deve registrar sobre ele?  
**Resposta:**

4. Em uma frase, descrevam a regra profissional usada nesta atividade para decidir tratamentos.  
**Resposta:**

## Critérios para avaliação da atividade

- Identifica ausências e possíveis outliers nos campos priorizados.
- Usa regras de negócio e verificações operacionais como evidência.
- Justifica decisões sem aplicar exclusão, preenchimento ou correção automáticos.
- Aplica as intervenções em uma cópia e verifica consequências.
- Mantém explícitos os casos inconclusivos e documenta o que foi alterado.

---
**Sobre os dados:** conjunto simulado de alta fidelidade, produzido para esta atividade. As conclusões descrevem somente este conjunto didático.


---
# Expectativa de respostas

## Registro 1

1. `distancia_km` pode estar ausente porque **não se aplica** a pedidos de retirada; já `tempo_separacao_min` ausente indica ausência de uma medição necessária ao processo e precisa ser investigada.  
2. Um valor muito alto pode representar erro, ocorrência legítima ou situação ainda inconclusiva; a magnitude, sozinha, não determina exclusão.

## Registro 2 — ausências

- `desconto_pct` ausente com `cupom_aplicado = Não`: **PREENCHER_ZERO**, porque a regra comercial documentada define 0% nesse contexto. Há 12 ocorrências.
- `distancia_km` em `Retirada`: **MANTER_AUSENTE**; o campo não se aplica. Há 32 pedidos de retirada.
- `avaliacao_cliente_1a5` ausente: **MANTER_AUSENTE**; significa ausência de resposta à pesquisa, não nota zero. Há 50 ausências.
- `PED-2026-0056`: **CORRIGIR_VALOR_CONFIRMADO** para 37,0 min.
- `PED-2026-0094`: **MANTER_AUSENTE**; o valor correto não é recuperável e as demais informações do pedido continuam válidas.

## Registro 3 — tabela de decisões e plano esperado

A tabela deve estar completa antes da aplicação. Cada ação deve estar vinculada à evidência consultada e a uma justificativa. O log automático registra somente as intervenções efetivamente executadas e complementa — não substitui — esse registro.


| Item | Ação esperada | Núcleo da justificativa |
|---|---|---|
| REGRA_DESCONTO_SEM_CUPOM | PREENCHER_ZERO | Regra documentada; não é estimativa. |
| REGRA_DISTANCIA_RETIRADA | MANTER_AUSENTE | Não se aplica à retirada. |
| REGRA_AVALIACAO_SEM_RESPOSTA | MANTER_AUSENTE | Pesquisa opcional; ausência não é zero. |
| PED-2026-0056 | CORRIGIR_VALOR_CONFIRMADO | WMS confirma 37,0 min. |
| PED-2026-0094 | MANTER_AUSENTE | Valor não recuperável; preservar o restante do registro. |
| PED-2026-0047 | CORRIGIR_VALOR_CONFIRMADO | WMS confirma 58,0 min. |
| PED-2026-0112 | CORRIGIR_VALOR_CONFIRMADO | Nota fiscal confirma 9 itens. |
| PED-2026-0159 | MANTER | 244,0 min é ocorrência legítima e documentada. |
| PED-2026-0136 | TRATAR_COMO_AUSENTE | 410,0 min é inválido e o correto não é recuperável. |
| PED-2026-0171 | INVESTIGAR_MAIS | Evidência insuficiente para confirmar erro ou legitimidade. |

## Conferência computacional

Antes do tratamento, os candidatos pelo critério de 1,5 × AIQ são:

- `tempo_separacao_min`: PED-2026-0136, PED-2026-0047, PED-2026-0171, PED-2026-0159;
- `itens_qtd`: PED-2026-0112.

Após o plano de referência:

- `tempo_separacao_min`: PED-2026-0171, PED-2026-0159;
- `itens_qtd`: PED-2026-0112.

A média de `tempo_separacao_min` passa de **40.56** para **35.53** min; a mediana passa de **33.20** para **33.20** min.

O resultado esperado **não é eliminar todos os candidatos a outlier nem todas as ausências**. Valores legítimos ou ainda inconclusivos podem permanecer, e ausências estruturais/sem resposta devem continuar ausentes.

## Registro 4

1. Devem permanecer, entre outras, `distancia_km` nas retiradas, avaliações não respondidas e o tempo de `PED-2026-0094`. Preenchê-las automaticamente criaria informação que não existe ou apagaria o significado da ausência.  
2. Permanecem candidatos legítimos ou inconclusivos, como `PED-2026-0159` e `PED-2026-0171`; o valor corrigido de itens pode continuar estatisticamente atípico, mas está confirmado por fonte operacional.  
3. `PED-2026-0171` permanece inconclusivo. A equipe deve registrar a pendência e não alterar o valor sem evidência.  
4. Formulação esperada: **a decisão de tratamento depende da evidência, do contexto e do impacto na análise, não apenas do tamanho do valor ou da presença de uma célula vazia.**
